# Aviation Trends
**Last Updated: April 26th 2026**

This notebook analyses daily aviation data to and from - Dubai, Doha, Amman, Cairo, Casablanca, Riyadh, Jeddah and Abu Dhabi. The main analyses involves daily and weekly changes in flights departed and arrived. Additionally, we look at the destinations where there were the most number of flights changes. 

## Data 

The dataset used for this update is from [Aviation Stack](https://aviationstack.com/documentation).This data is purchased through their API subscription. The data is being validated for Dubai against OAG aggregated dataset from April 2025-June 2025. 


In [1]:
import pandas as pd
from utils import *

Define a function to get a range of dates for the desirable date range

In [2]:
# Define empty dataset to concat all the arrivals and departures
departures = pd.DataFrame()
arrivals = pd.DataFrame()


In [3]:
import glob
from utils import *

arrivals = pd.DataFrame()

for file in glob.glob('../../data/aviation/arrivals_*.csv'):
    df = pd.read_csv(file)
    arrivals = pd.concat([arrivals,df])

#arrivals.drop(columns="Unnamed: 0", inplace=True)
arrivals.drop_duplicates(inplace=True)
arrivals.reset_index(drop=True, inplace=True)

for file in glob.glob('../../data/aviation/departures_*.csv'):
    df = pd.read_csv(file)
    departures = pd.concat([departures,df])

#departures.drop(columns="Unnamed: 0", inplace=True)
#
# departures.drop_duplicates(inplace=True)
departures.reset_index(drop=True, inplace=True)

check_missing_dates(arrivals)
#check_missing_dates(departures)

All dates are present.


In [4]:
airports = ['DXB', 'DOH', 'AMM', 'JED', 'AUH', 'RUH', 'CAI', 'CMN'] + ['TUN', 'BEY', 'KHI', 'DAM','IKA', 'ALG', 'BGW']

In [5]:
# Check if the number of flights per day are 100. If they are exactly 100 we need to rerun the API to get the next 100 flight.
departures["flight_date"].value_counts()

flight_date
2026-02-14    7495
2026-02-08    7443
2026-02-15    7433
2026-02-07    7411
2026-02-13    7403
              ... 
2025-09-30      13
2025-10-09      12
2025-09-22      10
2025-09-14       9
2025-11-10       7
Name: count, Length: 368, dtype: int64

In [6]:
# Check if the number of flights per day are 100. If they are exactly 100 we need to rerun the API to get the next 100 flight.
arrivals.sort_values(by="flight_date", ascending=False)["flight_date"].value_counts()

flight_date
2026-02-15    7408
2026-02-14    7406
2026-02-08    7405
2026-02-07    7362
2026-02-01    7338
              ... 
2026-03-30    4049
2026-03-27    3995
2026-04-07    3975
2026-03-31    3874
2026-04-23    2892
Name: count, Length: 116, dtype: int64

Functions to clean the database and explode columns

In [7]:
import ast


def safe_literal_eval(value):
    if isinstance(value, str):
        try:
            return ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return value
    return value


def explode(flights):
    flights["arrival"] = flights["arrival"].apply(safe_literal_eval)
    flights["departure"] = flights["departure"].apply(safe_literal_eval)

    fr1 = pd.json_normalize(flights["arrival"]).add_suffix("_arr")
    fr2 = pd.json_normalize(flights["departure"]).add_suffix("_dep")

    flights_exploded = pd.concat(
        [flights.drop(columns=["arrival", "departure"]), fr1, fr2], axis=1
    )

    return flights_exploded

In [8]:
%load_ext autoreload
%autoreload 2
from visuals import *

In [9]:
departures_exploded = explode(departures)
# departures_explode
# d = departures_exploded[~(departures_exploded['airport_arr']==departures_exploded['airport_dep'])]

In [10]:
check_missing_dates(departures)

Flights for the following dates are missing in the data, reasons are unknown.: DatetimeIndex(['2025-06-22'], dtype='datetime64[ns]', freq='D')


## Arrival Trends in Major Airports in MENAAP

In [11]:
beginning = departures["flight_date"].min()
end = departures["flight_date"].max()
print(f"Data is available from {beginning} to {end}")

Data is available from 2025-04-23 to 2026-04-26


Conduct a duplication check for flights. If the flight is taking off from the same place, to the same place at the same time and has two entries, it is a duplicate flight

In [12]:
before = departures_exploded.shape[0]
print(f"There were {before} flights before duplication check")
# check for duplicate flights i.e., flights scheduled to take off at the exact same time from the same place to the same destination
departures_exploded = departures_exploded.drop_duplicates(
    subset=["flight_date", "scheduled_arr", "iata_arr", "iata_dep", "scheduled_dep"]
)

after = departures_exploded.shape[0]
print(
    f"There are {after} flights after duplication check. {before-after} flights were duplicated"
)

There were 714306 flights before duplication check
There are 319876 flights after duplication check. 394430 flights were duplicated


### Flight Status Legend
- Scheduled: A flight that we have a schedule or flight plan for that hasn’t departed or has been canceled.
- Active: A flight that either left the gate or the runway and is on its way to its destination.
- Landed or Arrived: A flight that landed on the runway or arrived at the gate at the destination.
- Canceled: A flight that one or more data sources have indicated is canceled.
- Redirected: The flight is being redirected to another airport.
- Diverted: A flight that has landed or arrived at the gate of an airport where it wasn’t scheduled to arrive.
- Unknown: We were unable to detect the final arrival status.

In [13]:
# Test to see if any flight has more than one flight status assoctaed with it.
duplicate_status_test = (
    departures_exploded.groupby(
        ["flight_date", "scheduled_arr", "iata_arr", "iata_dep", "scheduled_dep"]
    )[["flight_status"]]
    .count()
    .reset_index()
)
duplicate_status_test[duplicate_status_test["flight_status"] > 1]

,flight_date,scheduled_arr,iata_arr,iata_dep,scheduled_dep,flight_status


In [14]:
# missing_dates

events = {
    "2024-06-09": "Airline data\nnot available",
}

### Daily Departures by Flight Status

In [15]:
departures_exploded["flight_date"] = pd.to_datetime(departures_exploded["flight_date"])

df = (
    departures_exploded.groupby(["flight_date", "flight_status", "iata_dep"])
    .size()
    .reset_index(name="count")
)

get_area_plot_by_airport(
    df,
    airports=airports,
    airport_col="iata_dep",
    title="Daily Departures by Airport",
    source_text="Source: Flight data from AviationStack",
)

alt.FacetChart(...)

In [16]:
df = (
    departures_exploded.groupby(["flight_date", "flight_status", "iata_dep"])
    .size()
    .reset_index(name="count")
)

df.to_csv('../../data/aviation/processed/departures_daily_airport.csv')

### Weekly Departures by Flight Status

In [17]:
departures_exploded["flight_date"] = pd.to_datetime(departures_exploded["flight_date"])

df = (
    departures_exploded.groupby(
        [pd.Grouper(key="flight_date", freq="W"), "flight_status", "iata_dep"]
    )
    .size()
    .reset_index(name="count")
)

df.to_csv('../../data/aviation/processed/departures_weekly_airport.csv')

df = df[(df['flight_date']<'2026-03-15')&(df['flight_date']>'2026-02-01')]

get_area_plot_by_airport(
    df,
    airports=airports,
    airport_col="iata_dep",
    title="Weekly Departures by Airport",
    source_text="Source: Flight data from AviationStack",
    reindex_freq="W",
)

alt.FacetChart(...)

In [18]:
departures_exploded["airportcity"] = departures_exploded["iata_arr"].map(iata_mapping)
departures_poi = departures_exploded[departures_exploded["flight_date"] >= "2026-02-28"]

# Build a single DataFrame with top-20 destinations per departure airport per status category
records = []
for iata_dep, group in departures_poi.groupby("iata_dep"):
    for category, statuses in {
        "Most Changed": ["scheduled", "cancelled", "diverted"],
        "Most Unchanged": ["landed"],
    }.items():
        top20 = (
            group[group["flight_status"].isin(statuses)]["airportcity"]
            .value_counts()
            .head(20)
            .reset_index()
        )
        top20["category"] = category
        top20["iata_dep"] = iata_dep
        records.append(top20)

departures_top10 = pd.concat(records, ignore_index=True)
departures_top10.to_csv('../../data/aviation/processed/departures_top20_changes.csv')

In [19]:
import altair as alt

airport_codes = sorted(departures_top10["iata_dep"].unique())

dropdown = alt.binding_select(options=airport_codes, name="Departure Airport ")
selection = alt.selection_point(fields=["iata_dep"], bind=dropdown, value=airport_codes[0])

colors = {"Most Changed": "#34A7F2", "Most Unchanged": "#FF9800"}

subplots = []
for category, color in colors.items():
    subset = departures_top10[departures_top10["category"] == category]

    chart = (
        alt.Chart(subset)
        .mark_bar(color=color, opacity=0.7)
        .encode(
            y=alt.Y("airportcity:N", sort="-x", title=None),
            x=alt.X("count:Q", title=None, axis=None),
            tooltip=[
                alt.Tooltip("airportcity:N", title="Destination"),
                alt.Tooltip("count:Q", title="Flights"),
            ],
        )
        .properties(width=150, height=400, title=category)
        .add_params(selection)
        .transform_filter(selection)
    )

    text = chart.mark_text(align="left", dx=3, fontSize=11, font="Open Sans").encode(
        text="count:Q"
    )

    subplots.append(chart + text)

final = (
    alt.hconcat(subplots[0], subplots[1])
    .properties(
        title=alt.Title(
            "Top 20 Destinations by Departure Status after 28th February 2026",
            subtitle="Source: AviationStack",
        )
    )
    .configure_title(font="Open Sans", subtitleFont="Open Sans")
    .configure_axis(labelFont="Open Sans", titleFont="Open Sans")
    .configure_legend(labelFont="Open Sans", titleFont="Open Sans")
    .configure_text(font="Open Sans")
)

final

alt.HConcatChart(...)

## Arrival Trends in Major Airports in MENAAP

In [20]:
check_missing_dates(arrivals)

All dates are present.


In [21]:
arrivals_exploded = explode(arrivals)

In [22]:
before = arrivals_exploded.shape[0]
print(f"There were {before} flights before duplication check")
# check for duplicate flights i.e., flights scheduled to take off at the exact same time from the same place to the same destination
arrivals_exploded = arrivals_exploded.drop_duplicates(
    subset=["flight_date", "scheduled_arr", "iata_arr", "iata_dep", "scheduled_dep"]
)

after = arrivals_exploded.shape[0]
print(
    f"There are {after} flights after duplication check. {before-after} flights were duplicated"
)

There were 682731 flights before duplication check
There are 306762 flights after duplication check. 375969 flights were duplicated


In [23]:
# Test to see if any flight has more than one flight status assoctaed with it.
duplicate_status_test = (
    arrivals_exploded.groupby(
        ["flight_date", "scheduled_arr", "iata_arr", "iata_dep", "scheduled_dep"]
    )[["flight_status"]]
    .count()
    .reset_index()
)
duplicate_status_test[duplicate_status_test["flight_status"] > 1]

,flight_date,scheduled_arr,iata_arr,iata_dep,scheduled_dep,flight_status


### Daily Number of Flights by Flight Status

In [24]:
arrivals_exploded["flight_date"] = pd.to_datetime(arrivals_exploded["flight_date"])

df = (
    arrivals_exploded.groupby(["flight_date", "flight_status", "iata_arr"])
    .size()
    .reset_index(name="count")
)
df.to_csv('../../data/aviation/processed/arrivals_daily_airport.csv')
get_area_plot_by_airport(
    df,
    airports=airports,
    airport_col="iata_arr",
    title="Daily Arrivals by Airport",
    source_text="Source: Flight data from AviationStack",
)

alt.FacetChart(...)

### Weekly Arrivals by Flight Status

In [25]:
arrivals_exploded["flight_date"] = pd.to_datetime(arrivals_exploded["flight_date"])

df = (
    arrivals_exploded.groupby(
        [pd.Grouper(key="flight_date", freq="W"), "flight_status", "iata_arr"]
    )
    .size()
    .reset_index(name="count")
)

df.to_csv('../../data/aviation/processed/arrivals_weekly_airport.csv')

df = df[df['flight_date']<'2026-03-15']

get_area_plot_by_airport(
    df,
    airports=airports,
    airport_col="iata_arr",
    title="Weekly Arrivals by Airport",
    source_text="Source: Flight data from AviationStack",
    reindex_freq="W",
)

alt.FacetChart(...)

In [26]:
arrivals_exploded["airportcity"] = arrivals_exploded["iata_dep"].map(iata_mapping)
arrivals_poi = arrivals_exploded[arrivals_exploded["flight_date"] >= "2026-02-28"]

# Build a single DataFrame with top-10 origins per arrival airport per status category
records = []
for iata_arr, group in arrivals_poi.groupby("iata_arr"):
    for category, statuses in {
        "Most Changed": ["scheduled", "cancelled", "diverted"],
        "Most Unchanged": ["landed", "active"],
    }.items():
        top10 = (
            group[group["flight_status"].isin(statuses)]["airportcity"]
            .value_counts()
            .head(20)
            .reset_index()
        )
        top10["category"] = category
        top10["iata_arr"] = iata_arr
        records.append(top10)

arrivals_top10 = pd.concat(records, ignore_index=True)
arrivals_top10.to_csv('../../data/aviation/processed/arrivals_top20_changes.csv')

### Origins with high and low disruptions

In [27]:
import altair as alt

airport_codes = sorted(arrivals_top10["iata_arr"].unique())

dropdown = alt.binding_select(options=airport_codes, name="Arrival Airport ")
selection = alt.selection_point(fields=["iata_arr"], bind=dropdown, value=airport_codes[0])

colors = {"Most Changed": "#34A7F2", "Most Unchanged": "#FF9800", }

subplots = []
for category, color in colors.items():
    subset = arrivals_top10[arrivals_top10["category"] == category]

    chart = (
        alt.Chart(subset)
        .mark_bar(color=color, opacity=0.7)
        .encode(
            y=alt.Y("airportcity:N", sort="-x", title=None),
            x=alt.X("count:Q", title=None, axis=None),
            tooltip=[
                alt.Tooltip("airportcity:N", title="Origin"),
                alt.Tooltip("count:Q", title="Flights"),
            ],
        )
        .properties(width=180, height=400, title=category)
        .add_params(selection)
        .transform_filter(selection)
    )

    text = chart.mark_text(align="left", dx=3, fontSize=11, font="Open Sans").encode(
        text="count:Q"
    )

    subplots.append(chart + text)

final = (
    alt.hconcat(subplots[0], subplots[1])
    .properties(
        title=alt.Title(
            "Top 20 Origins by Arrival Status after 28th February 2026",
            subtitle="Source: AviationStack | Changed flights = scheduled + cancelled + diverted | Unchanged flights = landed + active",
        )
    )
    .configure_title(font="Open Sans", subtitleFont="Open Sans")
    .configure_axis(labelFont="Open Sans", titleFont="Open Sans")
    .configure_legend(labelFont="Open Sans", titleFont="Open Sans")
    .configure_text(font="Open Sans")
)

final

alt.HConcatChart(...)

In [28]:
departures_exploded.to_csv('../../data/aviation/processed/departures_exploded_2026_01_01-2026_03_10.csv')

In [29]:
arrivals_exploded.to_csv('../../data/aviation/processed/arrivals_exploded_2026_01_01-2026_03_10.csv')

## Validation with OAG

In [30]:
oag_dxb_arr = pd.read_excel('../../data/oag/Arrivals_MENA_air_JobId3639446.xlsx', skiprows=24)
oag_dxb_arr = oag_dxb_arr[oag_dxb_arr['Arr Airport Name']=='Dubai International'].sort_values(by='Time series')

/Users/ssarva/Library/CloudStorage/OneDrive-WBG/Documents/MENA-FCV-economic-monitor/.venv/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [31]:
oag_dxb_dep = pd.read_excel('../../data/oag/Departures_MENA_air_JobId3639447.xlsx', skiprows=24)
oag_dxb_dep = oag_dxb_dep[oag_dxb_dep['Dep Airport Name']=='Dubai International'].sort_values(by='Time series')

/Users/ssarva/Library/CloudStorage/OneDrive-WBG/Documents/MENA-FCV-economic-monitor/.venv/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [32]:
# dxb_arr = pd.concat([
#     pd.read_csv('../../data/aviation/arrivals_DXB_2025-04-01_2025-06-30.csv')
# ])

# dxb_dep = pd.concat([
#     pd.read_csv('../../data/aviation/departures_DXB_2025-12-01_2025-12-31.csv')
# ])

In [33]:
# dxb_arr.reset_index(drop=True, inplace=True)
# dxb_arr_exploded = explode(dxb_arr)
# dxb_arr_exploded['flight_date'] = pd.to_datetime(dxb_arr_exploded['flight_date'])

# dxb_dep.reset_index(drop=True, inplace=True)
# dxb_dep_exploded = explode(dxb_dep)
# dxb_dep_exploded['flight_date'] = pd.to_datetime(dxb_dep_exploded['flight_date'])

In [34]:
#dxb_dep_exploded['flight_date'].unique()

In [35]:
# before = dxb_arr_exploded.shape[0]
# print(f"There were {before} flights before duplication check")
# # check for duplicate flights i.e., flights scheduled to take off at the exact same time from the same place to the same destination
# dxb_arr_exploded = dxb_arr_exploded.drop_duplicates(
#     subset=["flight_date", "scheduled_arr", "iata_arr", "iata_dep", "scheduled_dep"]
# )

# after = dxb_arr_exploded.shape[0]
# print(
#     f"There are {after} flights after duplication check. {before-after} flights were duplicated"
# )

# before = dxb_dep_exploded.shape[0]
# print(f"There were {before} flights before duplication check")
# # check for duplicate flights i.e., flights scheduled to take off at the exact same time from the same place to the same destination
# dxb_dep_exploded = dxb_dep_exploded.drop_duplicates(
#     subset=["flight_date", "scheduled_arr", "iata_arr", "iata_dep", "scheduled_dep"]
# )

# after = dxb_dep_exploded.shape[0]
# print(
#     f"There are {after} flights after duplication check. {before-after} flights were duplicated"
# )

In [36]:
# dxb_arr_grouped = dxb_arr_exploded[dxb_arr_exploded['iata_arr']=='DXB'].groupby(pd.Grouper(key='flight_date', freq='MS'))[['flight_status']].count().reset_index()

In [37]:
# dxb_dep_grouped = dxb_dep_exploded[dxb_dep_exploded['iata_dep']=='DXB'].groupby(pd.Grouper(key='flight_date', freq='MS'))[['flight_status']].count().reset_index()